In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.types import (StructType, StructField, IntegerType, StringType)

spark = (
    SparkSession.builder
    .appName("bdpa-clickflow-eclothing-recommender")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .config("spark.memory.fraction", "0.6")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

raw = (
    spark.read
        .option("sep", ";")
        .option("header", "true")
        .csv("../dataset/e_shop_clothing_2008.csv")
        .toDF(
              "year", "month", "day", "order", "country",
        "session_id", "category", "item_id", "colour",
        "location", "photo_type", "price", "price_above_avg", "page"
        )
        .withColumn("year",            F.col("year").cast(IntegerType()))
        .withColumn("month",           F.col("month").cast(IntegerType()))
        .withColumn("day",             F.col("day").cast(IntegerType()))
        .withColumn("order",           F.col("order").cast(IntegerType()))
        .withColumn("country",         F.col("country").cast(IntegerType()))
        .withColumn("session_id",      F.col("session_id").cast(IntegerType()))
        .withColumn("category",        F.col("category").cast(IntegerType()))
        .withColumn("colour",          F.col("colour").cast(IntegerType()))
        .withColumn("location",        F.col("location").cast(IntegerType()))
        .withColumn("photo_type",      F.col("photo_type").cast(IntegerType()))
        .withColumn("price",           F.col("price").cast(IntegerType()))
        .withColumn("price_above_avg", F.col("price_above_avg").cast(IntegerType()))
        .withColumn("page",            F.col("page").cast(IntegerType()))
) 
    

assert raw.count() == 165474
assert raw.filter(F.col("session_id").isNull()).count() == 0
assert raw.filter(F.col("item_id").isNull()).count() == 0
assert raw.select("month").distinct().count() == 5 # april to august
assert raw.select("category").distinct().count() == 4 # trousers, skirts, blouses, sale
assert raw.select("item_id").distinct().count() == 217


print("All validation checks passed.")
raw.printSchema()
raw.show(5)



All validation checks passed.
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- category: integer (nullable = true)
 |-- item_id: string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- photo_type: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price_above_avg: integer (nullable = true)
 |-- page: integer (nullable = true)

+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|year|month|day|order|country|session_id|category|item_id|colour|location|photo_type|price|price_above_avg|page|
+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|2008|    4|  1|    1|     29|         1|       1|    A13|     1|  

In [11]:
session_lengths = (
    raw.groupBy("session_id")
        .agg(F.count("*").alias("n_clicks"))
)

session_lengths.describe("n_clicks").show()

n_single = session_lengths.filter(F.col("n_clicks") == 1).count()

n_total = session_lengths.count() 
print(f"Single click session: {n_single}/{n_total} ({(n_single/n_total * 100)} %)")


# item popularity

item_freq = (
    raw.groupBy('item_id') 
        .agg(F.count("*").alias("views"))
        .orderBy(F.desc("views"))
)

item_freq.show(10)


n_sessions = raw.select("session_id").distinct().count() 
n_items = raw.select("item_id").distinct().count() 
n_pairs = raw.select("session_id", "item_id").distinct().count() 
sparsity = 1 - n_pairs / (n_sessions * n_items)

print(f"Utility matrix: {n_sessions} x {n_items}")
print(f"Observed pairs: {n_pairs}")
print(f"Sparsity: {sparsity:.4%}")


# temporal distribution 
raw.groupBy("month").count().orderBy("month").show() 

raw.groupBy("category").count().orderBy("category").show()

(
    raw.groupBy("session_id", "country")
    .count() 
    .groupBy("country")
    .agg(F.countDistinct("session_id").alias("n_sessions"))
    .orderBy(F.desc("n_sessions"))
    .show(10)
)




+-------+------------------+
|summary|          n_clicks|
+-------+------------------+
|  count|             24026|
|   mean|6.8872887704986265|
| stddev|   8.9951606320672|
|    min|                 1|
|    max|               195|
+-------+------------------+

Single click session: 5042/24026 (20.985598934487637 %)
+-------+-----+
|item_id|views|
+-------+-----+
|     B4| 3579|
|     A2| 3013|
|    A11| 2789|
|     P1| 2681|
|    B10| 2566|
|     A4| 2522|
|    A15| 2489|
|     A5| 2354|
|    A10| 2280|
|     A1| 2265|
+-------+-----+
only showing top 10 rows
Utility matrix: 24026 x 217
Observed pairs: 148345
Sparsity: 97.1547%
+-----+-----+
|month|count|
+-----+-----+
|    4|48199|
|    5|35654|
|    6|32242|
|    7|35231|
|    8|14148|
+-----+-----+

+--------+-----+
|category|count|
+--------+-----+
|       1|49742|
|       2|38408|
|       3|38577|
|       4|38747|
+--------+-----+

+-------+----------+
|country|n_sessions|
+-------+----------+
|     29|     19582|
|      9|      

In [12]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window 

w_desc = Window.partitionBy("session_id").orderBy(F.desc("order"))

raw_with_rank = raw.withColumn("click_rank", F.row_number().over(w_desc))

ground_truth_full = (
    raw_with_rank
    .filter(F.col("click_rank") == 1)
    .select("session_id", "item_id")
)

train_raw_full = raw_with_rank.filter(F.col("click_rank") > 1)

session_train_stats = (
    train_raw_full.groupBy("session_id")
        .agg(F.count("*").alias("n_train_clicks"))
)

valid_sessions = (
    session_train_stats
    .filter(F.col("n_train_clicks") >= 2)
    .select("session_id")
)

cold_sessions = (
    session_train_stats
    .filter(F.col("n_train_clicks") == 1)
    .select("session_id")
)

train_raw = train_raw_full.join(valid_sessions, on="session_id", how="inner")
ground_truth = ground_truth_full.join(valid_sessions, on="session_id", how="inner")

cold_train_raw = (
    train_raw_full
    .join(cold_sessions, on="session_id", how="inner")
    .select("session_id", "item_id", "order")
)

cold_ground_truth = (
    ground_truth_full
    .join(cold_sessions, on="session_id", how="inner")
    .select("session_id", "item_id")
)

print(f"Warm sessions in training: {train_raw.select('session_id').distinct().count()}")
print(f"Warm ground truth pairs:   {ground_truth.count()}")
print(f"Cold-start sessions (1 observed click): {cold_sessions.count()}")
print(f"Cold ground-truth pairs:               {cold_ground_truth.count()}")



Warm sessions in training: 15664
Warm ground truth pairs:   15664
Cold-start sessions (1 observed click): 3320
Cold ground-truth pairs:               3320


In [13]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS 

ALPHA = 80.0

interactions = (
    train_raw
        .groupBy("session_id", "item_id") 
        .agg(F.count("*").alias("click_count"))
)

item_indexer = StringIndexer(inputCol='item_id', outputCol="item_idx").fit(interactions)

train_idx = (
    item_indexer.transform(interactions)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
)

train_idx.cache()

ground_truth_idx = (
    item_indexer.transform(ground_truth)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
    .select("session_idx", "item_idx")
)

als = ALS(
    rank = 20,
    maxIter=15,
    regParam=1,
    alpha=ALPHA,
    implicitPrefs=True,
    userCol="session_idx",
    itemCol="item_idx",
    ratingCol="click_count",
    coldStartStrategy="drop",
    seed=42
)

als_model = als.fit(train_idx)

all_sessions = train_idx.select("session_idx").distinct()
raw_recs = als_model.recommendForUserSubset(all_sessions, numItems=10)

recs = (
    raw_recs
    .select(
        F.col("session_idx"),
        F.posexplode("recommendations").alias("rank_0", "rec")
    )
    .select(
        "session_idx",
        (F.col("rank_0") + 1).alias("rank"),
        F.col("rec.item_idx").alias("item_idx"),
        F.col("rec.rating").alias("score"),
    )
)

print(f"Recommendation rows: {recs.count()}")
recs.show(10)



Recommendation rows: 156640


+-----------+----+--------+----------+
|session_idx|rank|item_idx|     score|
+-----------+----+--------+----------+
|          1|   1|      31| 1.0824773|
|          1|   2|      29| 1.0234143|
|          1|   3|      79|0.97137964|
|          1|   4|       6| 0.9692417|
|          1|   5|     135|0.94846284|
|          1|   6|      17|0.94593084|
|          1|   7|       0| 0.9450717|
|          1|   8|     143|0.90349835|
|          1|   9|     129|0.88752854|
|          1|  10|      28|0.87011087|
+-----------+----+--------+----------+
only showing top 10 rows


In [14]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator 

param_grid = (
    ParamGridBuilder()
    .addGrid(als.rank, [20, 50, 100])
    .addGrid(als.regParam, [0.01, 0.1, 1.0])
    .addGrid(als.alpha, [10.0, 40.0, 80.0])
    .build()
)

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="click_count",
    predictionCol="prediction"
)

cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42
)

#cv_model = cv.fit(train_idx)
#best_model = cv_model.bestModel

#print("Best rank:     ", best_model.rank)
#print("Best regParam: ", best_model._java_obj.parent().getRegParam())
#print("Best alpha:    ", best_model._java_obj.parent().getAlpha())



In [15]:
item_features = (
    raw.groupBy("item_id","category","colour","photo_type","price_above_avg")
       .agg(F.mean("price").alias("avg_price"))
       .distinct()
)

from pyspark.ml.feature import VectorAssembler, OneHotEncoder

ohe = OneHotEncoder(
    inputCols=["category","colour","photo_type","price_above_avg"],
    outputCols=["cat_vec","col_vec","photo_vec","price_vec"],
    dropLast=True,
)

assembler = VectorAssembler(
    inputCols=["cat_vec","col_vec","photo_vec","price_vec","avg_price"],
    outputCol="content_features"
)

item_content = assembler.transform(ohe.fit(item_features).transform(item_features))
item_content.select("item_id","content_features").show(5, truncate=False)

+-------+-----------------------------------+
|item_id|content_features                   |
+-------+-----------------------------------+
|C53    |(23,[3,19,22],[1.0,1.0,38.0])      |
|C40    |(23,[3,22],[1.0,28.0])             |
|B12    |(23,[2,6,19,22],[1.0,1.0,1.0,38.0])|
|P2     |(23,[7,19,22],[1.0,1.0,28.0])      |
|P11    |(23,[8,21,22],[1.0,1.0,38.0])      |
+-------+-----------------------------------+
only showing top 5 rows


In [16]:
def evaluate_recommendations(recs_df, ground_truth_df, k=10):
    top_k = recs_df.filter(F.col("rank") <= k)

    hits = (
        top_k.join(
            ground_truth_df.withColumn("hit", F.lit(1.0)),
            on=["session_idx","item_idx"],
            how="left"
        ).fillna(0, subset=["hit"])
    )

    n_rel = (
        ground_truth_df.groupBy("session_idx")
                       .agg(F.count("*").alias("n_relevant"))
    )

    dcg = (
        hits
        .withColumn("gain",
                    (F.pow(F.lit(2), F.col("hit")) - 1) /
                    F.log2(F.col("rank").cast("double") + 1))
        .groupBy("session_idx")
        .agg(F.sum("gain").alias("dcg"))
    )

    idcg = (
        n_rel.withColumn("k_eff", F.least(F.col("n_relevant"), F.lit(k)))
             .withColumn("idcg",
                 F.expr("aggregate(sequence(1,k_eff),0D,"
                        "(a,i)->a+1.0/log2(cast(i+1 as double)))"))
    )

    ndcg_scores = (
        dcg.join(idcg, on="session_idx")
           .withColumn("ndcg",
                       F.when(F.col("idcg") > 0,
                              F.col("dcg") / F.col("idcg")).otherwise(0.0))
    )

    pr = (
        hits.groupBy("session_idx")
            .agg(F.sum("hit").alias("n_hits"))
            .join(n_rel, on="session_idx")
            .withColumn("precision", F.col("n_hits") / F.lit(k))
            .withColumn("recall",    F.col("n_hits") / F.col("n_relevant"))
    )

    results = {
        f"NDCG@{k}":      ndcg_scores.agg(F.mean("ndcg")).collect()[0][0],
        f"Precision@{k}": pr.agg(F.mean("precision")).collect()[0][0],
        f"Recall@{k}":    pr.agg(F.mean("recall")).collect()[0][0],
    }
    return results

item_popularity = (
    train_idx
    .groupBy("item_idx")
    .agg(F.sum("click_count").alias("total_clicks"))
    .orderBy(F.desc("total_clicks"), F.asc("item_idx"))
    .limit(10)
    .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("total_clicks"), F.asc("item_idx"))))
)

test_sessions = ground_truth_idx.select("session_idx").distinct()

pop_recs = (
    test_sessions
    .crossJoin(item_popularity.select("item_idx", "rank"))
)

pop_metrics = evaluate_recommendations(pop_recs, ground_truth_idx, k=10)
print("=== Popularity Baseline ===")
for metric_name, metric_value in pop_metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")

als_metrics = evaluate_recommendations(recs, ground_truth_idx, k=10)
print("=== ALS Model ===")
for metric_name, metric_value in als_metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")



26/04/25 23:11:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 2

=== Popularity Baseline ===
  NDCG@10: 0.0606
  Precision@10: 0.0121
  Recall@10: 0.1213


=== ALS Model ===
  NDCG@10: 0.1338
  Precision@10: 0.0277
  Recall@10: 0.2767


In [17]:
def diversity_metrics(recs_df, train_df, n_catalog_items=217):
    n_unique_recommended = recs_df.select("item_idx").distinct().count()
    coverage = n_unique_recommended / n_catalog_items

    n_sessions_train = train_df.select("session_idx").distinct().count()
    item_pop_fraction = (
        train_df
        .groupBy("item_idx")
        .agg(
            (F.countDistinct("session_idx") / F.lit(n_sessions_train)).alias("pop_fraction")
        )
    )


    return {
        "Coverage": coverage,
        "UniqueRecommendedItems": n_unique_recommended,
    }

als_div = diversity_metrics(recs, train_idx, n_catalog_items=217)
pop_div = diversity_metrics(pop_recs, train_idx, n_catalog_items=217)

print(f"Popularity Coverage: {pop_div['UniqueRecommendedItems']}/217 = {pop_div['Coverage']:.2%}")
print(f"ALS Coverage:        {als_div['UniqueRecommendedItems']}/217 = {als_div['Coverage']:.2%}")

def delta(a, b):
    return a - b

print(f"NDCG@10      baseline={pop_metrics['NDCG@10']:.4f}  ALS={als_metrics['NDCG@10']:.4f}  delta={delta(als_metrics['NDCG@10'], pop_metrics['NDCG@10']):+.4f}")
print(f"Precision@10 baseline={pop_metrics['Precision@10']:.4f}  ALS={als_metrics['Precision@10']:.4f}  delta={delta(als_metrics['Precision@10'], pop_metrics['Precision@10']):+.4f}")
print(f"Recall@10    baseline={pop_metrics['Recall@10']:.4f}  ALS={als_metrics['Recall@10']:.4f}  delta={delta(als_metrics['Recall@10'], pop_metrics['Recall@10']):+.4f}")
print(f"Coverage     baseline={pop_div['Coverage']:.2%}  ALS={als_div['Coverage']:.2%}  delta={delta(als_div['Coverage'], pop_div['Coverage']):+.2%}")



Popularity Coverage: 10/217 = 4.61%
ALS Coverage:        212/217 = 97.70%
NDCG@10      baseline=0.0606  ALS=0.1338  delta=+0.0732
Precision@10 baseline=0.0121  ALS=0.0277  delta=+0.0155
Recall@10    baseline=0.1213  ALS=0.2767  delta=+0.1554
Coverage     baseline=4.61%  ALS=97.70%  delta=+93.09%


In [18]:
# cold-start fix: session-sequential warm-up from first interaction

import math
from collections import defaultdict

spark.catalog.clearCache()

w_seq = Window.partitionBy("session_id").orderBy(F.asc("order"))

seq_pairs = (
    train_raw
    .select("session_id", "order", "item_id")
    .withColumn("next_item_id", F.lead("item_id").over(w_seq))
    .filter(F.col("next_item_id").isNotNull())
    .select(F.col("item_id").alias("seed_item_id"), "next_item_id")
)

trans_counts = (
    seq_pairs
    .groupBy("seed_item_id", "next_item_id")
    .agg(F.count("*").alias("pair_count"))
)

out_counts = (
    trans_counts
    .groupBy("seed_item_id")
    .agg(F.sum("pair_count").alias("out_total"))
)

trans_scores = (
    trans_counts
    .join(out_counts, on="seed_item_id", how="inner")
    .withColumn("score", F.col("pair_count") / F.col("out_total"))
)

w_first = Window.partitionBy("session_id").orderBy(F.asc("order"))

cold_seed = (
    cold_train_raw
    .withColumn("rn", F.row_number().over(w_first))
    .filter(F.col("rn") == 1)
    .select(F.col("session_id").cast("int").alias("session_idx"), F.col("item_id").alias("seed_item_id"))
)

seq_scored = (
    cold_seed
    .join(trans_scores, on="seed_item_id", how="left")
    .select("session_idx", F.col("next_item_id").alias("item_id"), "score")
    .filter(F.col("item_id").isNotNull())
)

seq_scored_idx = (
    item_indexer.transform(seq_scored)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .select("session_idx", "item_idx", "score")
    .dropna(subset=["item_idx"])
)

w_rank_seq = Window.partitionBy("session_idx").orderBy(F.desc("score"), F.asc("item_idx"))

seq_top10 = (
    seq_scored_idx
    .withColumn("rank", F.row_number().over(w_rank_seq))
    .filter(F.col("rank") <= 10)
    .select("session_idx", "rank", "item_idx", "score")
)

seq_counts = seq_top10.groupBy("session_idx").agg(F.count("*").alias("n_seq"))

cold_ground_truth_idx = (
    item_indexer.transform(cold_ground_truth)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
    .select("session_idx", "item_idx")
    .dropna(subset=["item_idx"])
)

need_fill = (
    cold_ground_truth_idx.select("session_idx").distinct()
    .join(seq_counts, on="session_idx", how="left")
    .fillna(0, subset=["n_seq"])
    .withColumn("n_missing", F.lit(10) - F.col("n_seq"))
    .filter(F.col("n_missing") > 0)
)

existing = seq_top10.select("session_idx", "item_idx").distinct()

fill_pool = (
    need_fill.select("session_idx", "n_missing")
    .crossJoin(item_popularity.select("item_idx", F.col("rank").alias("pop_rank")))
    .join(existing, on=["session_idx", "item_idx"], how="left_anti")
)

w_fill = Window.partitionBy("session_idx").orderBy(F.asc("pop_rank"), F.asc("item_idx"))

fill_recs = (
    fill_pool
    .withColumn("fill_order", F.row_number().over(w_fill))
    .filter(F.col("fill_order") <= F.col("n_missing"))
    .select(
        "session_idx",
        "item_idx",
        (F.lit(0.0) - F.col("pop_rank") * F.lit(1e-6)).alias("score")
    )
)

combined = seq_top10.select("session_idx", "item_idx", "score").unionByName(fill_recs)

w_rank_final = Window.partitionBy("session_idx").orderBy(F.desc("score"), F.asc("item_idx"))

cold_seq_recs = (
    combined
    .withColumn("rank", F.row_number().over(w_rank_final))
    .filter(F.col("rank") <= 10)
    .select("session_idx", "rank", "item_idx", "score")
)

cold_pop_recs = (
    cold_ground_truth_idx.select("session_idx").distinct()
    .crossJoin(item_popularity.select("item_idx", "rank"))
)

als_cold_sessions_covered = (
    recs.select("session_idx").distinct()
    .join(cold_ground_truth_idx.select("session_idx").distinct(), on="session_idx", how="inner")
    .count()
)

def evaluate_topk_pandas(recs_pdf, gt_pdf, k=10):
    gt_map = gt_pdf.groupby("session_idx")["item_idx"].apply(set).to_dict()
    recs_pdf = recs_pdf[recs_pdf["rank"] <= k].sort_values(["session_idx", "rank"])

    rec_map = defaultdict(list)
    for row in recs_pdf.itertuples(index=False):
        rec_map[row.session_idx].append(row.item_idx)

    ndcgs, precs, recs = [], [], []
    for sid, rel in gt_map.items():
        pred = rec_map.get(sid, [])
        hits = [1 if item in rel else 0 for item in pred]

        dcg = sum((1.0 / math.log2(r + 2)) for r, h in enumerate(hits) if h)
        idcg = 1.0  
        ndcgs.append(dcg / idcg)

        n_hits = sum(hits)
        precs.append(n_hits / float(k))
        recs.append(float(n_hits))

    return {
        f"NDCG@{k}": sum(ndcgs) / len(ndcgs) if ndcgs else 0.0,
        f"Precision@{k}": sum(precs) / len(precs) if precs else 0.0,
        f"Recall@{k}": sum(recs) / len(recs) if recs else 0.0,
    }


def diversity_from_pandas(recs_pdf, item_pop_fraction_map, n_catalog_items=217):
    unique_items = set(recs_pdf["item_idx"].tolist())
    coverage = len(unique_items) / float(n_catalog_items)

    return {
        "Coverage": coverage,
        "UniqueRecommendedItems": len(unique_items),
    }

seq_pdf = cold_seq_recs.select("session_idx", "rank", "item_idx").toPandas()
pop_pdf = cold_pop_recs.select("session_idx", "rank", "item_idx").toPandas()
gt_pdf = cold_ground_truth_idx.select("session_idx", "item_idx").toPandas()

n_sessions_train = train_idx.select("session_idx").distinct().count()
item_pop_fraction_pdf = (
    train_idx
    .groupBy("item_idx")
    .agg((F.countDistinct("session_idx") / F.lit(n_sessions_train)).alias("pop_fraction"))
    .toPandas()
)
item_pop_fraction_map = dict(zip(item_pop_fraction_pdf["item_idx"], item_pop_fraction_pdf["pop_fraction"]))

seq_metrics = evaluate_topk_pandas(seq_pdf, gt_pdf, k=10)
pop_metrics_cold = evaluate_topk_pandas(pop_pdf, gt_pdf, k=10)
seq_div = diversity_from_pandas(seq_pdf, item_pop_fraction_map, n_catalog_items=217)
pop_div_cold = diversity_from_pandas(pop_pdf, item_pop_fraction_map, n_catalog_items=217)

n_cold_sessions = gt_pdf["session_idx"].nunique()
n_seq_sessions = seq_pdf["session_idx"].nunique()

print("cold-start fix: Session-Sequential Warm-up ===")
print(f"Cold sessions evaluated: {n_cold_sessions}")
print(f"ALS session coverage on cold sessions: {als_cold_sessions_covered}/{n_cold_sessions}")
print(f"Sequential warm-up session coverage:   {n_seq_sessions}/{n_cold_sessions}")

print("\nPopularity baseline (cold sessions):")
for metric_name, metric_value in pop_metrics_cold.items():
    print(f"  {metric_name}: {metric_value:.4f}")
print(f"Coverage: {pop_div_cold['Coverage']:.2%}")

print("\nSequential warm-up (cold sessions):")
for metric_name, metric_value in seq_metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")
print(f"Coverage: {seq_div['Coverage']:.2%}")

print("\nDelta (Sequential - Baseline):")
print(f"NDCG@10:      {seq_metrics['NDCG@10'] - pop_metrics_cold['NDCG@10']:+.4f}")
print(f"Precision@10: {seq_metrics['Precision@10'] - pop_metrics_cold['Precision@10']:+.4f}")
print(f"Recall@10:    {seq_metrics['Recall@10'] - pop_metrics_cold['Recall@10']:+.4f}")
print(f"Coverage:     {seq_div['Coverage'] - pop_div_cold['Coverage']:+.2%}")

report_ready_metrics = {
    "cold_sessions": int(n_cold_sessions),
    "als_session_coverage": f"{als_cold_sessions_covered}/{n_cold_sessions}",
    "sequential_session_coverage": f"{n_seq_sessions}/{n_cold_sessions}",
    "baseline": {
        "NDCG@10": round(pop_metrics_cold["NDCG@10"], 4),
        "Precision@10": round(pop_metrics_cold["Precision@10"], 4),
        "Recall@10": round(pop_metrics_cold["Recall@10"], 4),
        "Coverage": round(pop_div_cold["Coverage"], 4),
    },
    "sequential_warmup": {
        "NDCG@10": round(seq_metrics["NDCG@10"], 4),
        "Precision@10": round(seq_metrics["Precision@10"], 4),
        "Recall@10": round(seq_metrics["Recall@10"], 4),
        "Coverage": round(seq_div["Coverage"], 4),
    },
    "delta_sequential_minus_baseline": {
        "NDCG@10": round(seq_metrics["NDCG@10"] - pop_metrics_cold["NDCG@10"], 4),
        "Precision@10": round(seq_metrics["Precision@10"] - pop_metrics_cold["Precision@10"], 4),
        "Recall@10": round(seq_metrics["Recall@10"] - pop_metrics_cold["Recall@10"], 4),
        "Coverage": round(seq_div["Coverage"] - pop_div_cold["Coverage"], 4),
    },
}

print("\nREPORT_READY_METRICS:")
print(report_ready_metrics)



26/04/25 23:11:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 23:11:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/25 2

cold-start fix: Session-Sequential Warm-up ===
Cold sessions evaluated: 3320
ALS session coverage on cold sessions: 0/3320
Sequential warm-up session coverage:   3320/3320

Popularity baseline (cold sessions):
  NDCG@10: 0.0918
  Precision@10: 0.0183
  Recall@10: 0.1831
Coverage: 4.61%

Sequential warm-up (cold sessions):
  NDCG@10: 0.2280
  Precision@10: 0.0439
  Recall@10: 0.4386
Coverage: 97.70%

Delta (Sequential - Baseline):
NDCG@10:      +0.1362
Precision@10: +0.0255
Recall@10:    +0.2554
Coverage:     +93.09%

REPORT_READY_METRICS:
{'cold_sessions': 3320, 'als_session_coverage': '0/3320', 'sequential_session_coverage': '3320/3320', 'baseline': {'NDCG@10': 0.0918, 'Precision@10': 0.0183, 'Recall@10': 0.1831, 'Coverage': 0.0461}, 'sequential_warmup': {'NDCG@10': 0.228, 'Precision@10': 0.0439, 'Recall@10': 0.4386, 'Coverage': 0.977}, 'delta_sequential_minus_baseline': {'NDCG@10': 0.1362, 'Precision@10': 0.0255, 'Recall@10': 0.2554, 'Coverage': 0.9309}}
